> **📋 Demo Note — Cell 1 (pre-run before audience arrives)**
>
> "Before the demo, we provisioned the infrastructure — MinIO for model storage, the serving runtime, and the service account. Think of this as the one-time platform setup any team would do."

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 1 — Deploy Infrastructure  (RHOAI 2.x)
# ══════════════════════════════════════════════════════════════════════════
#
# CLUSTER-ADMIN PRE-REQUISITES
# Run ALL of these once from a terminal as cluster-admin before this cell.
# Safe to re-run if namespace is recreated.
# ─────────────────────────────────────────────────────────────────────────
#
#   # 1. Grant workbench SA cluster-wide permissions
#   oc adm policy add-cluster-role-to-user self-provisioner system:serviceaccount:rhoai-demo:telco-wb
#   oc adm policy add-cluster-role-to-user cluster-reader system:serviceaccount:rhoai-demo:telco-wb
#
#   # 2. Create namespace
#   oc new-project telco-bias-demo 2>/dev/null || true
#
#   # 3. Grant workbench SA admin on the namespace
#   oc adm policy add-role-to-user admin system:serviceaccount:rhoai-demo:telco-wb -n telco-bias-demo
#
#   # 4. Allow workbench SA to create ServiceMonitors and PrometheusRules
#   #    (required for Cell 8b — Prometheus monitoring setup)
#   oc adm policy add-role-to-user monitoring-edit system:serviceaccount:rhoai-demo:telco-wb -n telco-bias-demo
#
#   # 5. Allow workbench SA to apply Grafana dashboard ConfigMap
#   #    (required for Cell 8b — applies dashboard to openshift-config-managed)
#   oc adm policy add-role-to-user admin system:serviceaccount:rhoai-demo:telco-wb -n openshift-config-managed
#
#   NOTE: No OSSM/Istio/service mesh configuration required in RHOAI 2.x.
#         KServe RawDeployment mode and TrustyAI work without service mesh.
#         Reference: https://github.com/dinlaks/trustyai-demos
#
#   # ── Verify pre-reqs ──────────────────────────────────────────────────
#   echo "=== Cluster rolebindings for telco-wb ==="
#   oc get clusterrolebinding | grep telco-wb
#
#   echo "=== Namespace ==="
#   oc get namespace telco-bias-demo
# ─────────────────────────────────────────────────────────────────────────

import subprocess, time, textwrap, base64, boto3, requests, warnings, json as _json
warnings.filterwarnings("ignore")

NAMESPACE        = "telco-bias-demo"
MODEL_NAME       = "network-slice-model"
BUCKET_NAME      = "models"
MINIO_ACCESS_KEY = "THEACCESSKEY"
MINIO_SECRET_KEY = "THESECRETKEY"
MINIO_ENDPOINT   = "http://minio.telco-bias-demo.svc.cluster.local:9000"
OVMS_IMAGE       = "quay.io/modh/openvino_model_server@sha256:6c7795279f9075bebfcd9aecbb4a4ce4177eec41fb3f3e1f1079ce6309b7ae45"

# ── Pre-flight: verify oc login ───────────────────────────────────────────────
try:
    whoami = subprocess.check_output(["oc","whoami"], stderr=subprocess.DEVNULL).decode().strip()
    print(f"Logged in as: {whoami}")
except subprocess.CalledProcessError:
    raise RuntimeError("Not logged in. Run: oc login <api-url> --token=<token>")
WB_SA = whoami

# ── Derive CLUSTER_DOMAIN ─────────────────────────────────────────────────────
import re as _re
CLUSTER_DOMAIN = ""
_s = subprocess.run(["oc","whoami","--show-server"], capture_output=True, text=True)
_m = _re.search(r"https?://api[.](.+?)(?:[:\d]+)?\s*$", _s.stdout.strip())
if _m:
    CLUSTER_DOMAIN = "apps." + _m.group(1)
if not CLUSTER_DOMAIN:
    _rc = subprocess.run(["oc","get","route","console","-n","openshift-console",
                          "-o","jsonpath={.spec.host}"], capture_output=True, text=True)
    _host = _rc.stdout.strip()
    if _host:
        _m3 = _re.search(r"[.]apps[.](.+)$", _host)
        if _m3:
            CLUSTER_DOMAIN = "apps." + _m3.group(1)
if not CLUSTER_DOMAIN:
    raise RuntimeError("Cannot derive CLUSTER_DOMAIN.")
if CLUSTER_DOMAIN.startswith("apps.apps."):
    CLUSTER_DOMAIN = CLUSTER_DOMAIN[5:]
print(f"Cluster domain: {CLUSTER_DOMAIN}")

# ── Helpers ───────────────────────────────────────────────────────────────────
def _apply(yaml_str, path="/tmp/_res.yaml"):
    with open(path, "w") as f:
        f.write(textwrap.dedent(yaml_str).strip() + "\n")
    r = subprocess.run(["oc","apply","-f",path], capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if r.returncode != 0:
        raise RuntimeError(f"oc apply failed:\n{out}")
    print(out)

def _wait_for(resource, name, ns, jsonpath, expected, timeout=600):
    print(f"Waiting for {resource}/{name} → '{expected}' (timeout={timeout}s)…")
    val = ""
    for i in range(timeout // 5):
        r = subprocess.run(["oc","get",resource,name,"-n",ns,"-o",
                            f"jsonpath={{{jsonpath}}}"], capture_output=True, text=True)
        val = r.stdout.strip()
        if i % 6 == 0:
            print(f"  [{i*5}s] current={val!r}")
        if val == expected:
            print(f"  ✅ {resource}/{name} is '{expected}'")
            return
        time.sleep(5)
    raise RuntimeError(f"Timeout {timeout}s — {resource}/{name} still {val!r}.")

def b64(s): return base64.b64encode(s.encode()).decode()

# ── 1. Namespace ──────────────────────────────────────────────────────────────
r = subprocess.run(["oc","new-project", NAMESPACE], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())
subprocess.run(["oc","adm","policy","add-role-to-user","admin",
                WB_SA, "-n", NAMESPACE], capture_output=True)
print(f"Admin granted: {WB_SA}")

# ── 2. MinIO — Deployment + PVC for persistent storage across env restarts ────
_apply(f"""
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: minio-pvc
  namespace: {NAMESPACE}
spec:
  accessModes: [ReadWriteOnce]
  resources:
    requests:
      storage: 2Gi
---
apiVersion: v1
kind: Service
metadata:
  name: minio
  namespace: {NAMESPACE}
spec:
  ports:
  - name: api
    port: 9000
    targetPort: 9000
  selector:
    app: minio
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: minio
  namespace: {NAMESPACE}
spec:
  replicas: 1
  selector:
    matchLabels:
      app: minio
  template:
    metadata:
      labels:
        app: minio
    spec:
      containers:
      - name: minio
        image: quay.io/trustyai_testing/modelmesh-minio-examples:latest
        args: [server, /data1]
        env:
        - {{name: MINIO_ACCESS_KEY, value: "{MINIO_ACCESS_KEY}"}}
        - {{name: MINIO_SECRET_KEY, value: "{MINIO_SECRET_KEY}"}}
        ports:
        - containerPort: 9000
        readinessProbe:
          tcpSocket: {{port: 9000}}
          initialDelaySeconds: 5
          periodSeconds: 5
        volumeMounts:
        - name: minio-storage
          mountPath: /data1
      volumes:
      - name: minio-storage
        persistentVolumeClaim:
          claimName: minio-pvc
""")
_wait_for("deployment","minio",NAMESPACE,".status.readyReplicas","1",timeout=120)

# ── 3. MinIO TLS Route + S3 Secret + KServe SA ───────────────────────────────
# KServe storage-initializer always uses HTTPS for S3 (ignores s3-usehttps=0).
# Create a TLS Edge Route for MinIO so the storage-initializer can connect via HTTPS.
_route_exists = subprocess.run(
    ["oc","get","route","minio-s3","-n",NAMESPACE],
    capture_output=True, text=True).returncode == 0
if not _route_exists:
    subprocess.run(
        ["oc","create","route","edge","minio-s3",
         f"--service=minio","--port=api",
         "--insecure-policy=Redirect",f"-n",NAMESPACE],
        capture_output=True, text=True)
    print("MinIO TLS edge route created")
else:
    print("MinIO TLS edge route already exists")

MINIO_ROUTE_HOST = subprocess.run(
    ["oc","get","route","minio-s3","-n",NAMESPACE,
     "-o","jsonpath={.spec.host}"],
    capture_output=True, text=True).stdout.strip()
print(f"MinIO route: {MINIO_ROUTE_HOST}")

_apply(f"""
apiVersion: v1
kind: Secret
metadata:
  name: aws-connection-minio-data-connection
  namespace: {NAMESPACE}
  labels:
    opendatahub.io/dashboard: "true"
    opendatahub.io/managed: "true"
  annotations:
    opendatahub.io/connection-type: s3
    openshift.io/display-name: Minio Data Connection
data:
  AWS_ACCESS_KEY_ID:     {b64(MINIO_ACCESS_KEY)}
  AWS_SECRET_ACCESS_KEY: {b64(MINIO_SECRET_KEY)}
  AWS_DEFAULT_REGION:    {b64("us-south")}
  AWS_S3_ENDPOINT:       {b64(MINIO_ROUTE_HOST)}
  AWS_S3_BUCKET:         {b64(BUCKET_NAME)}
type: Opaque
---
apiVersion: v1
kind: ServiceAccount
metadata:
  name: kserve-minio-sa
  namespace: {NAMESPACE}
  annotations:
    serving.kserve.io/s3-endpoint:          {MINIO_ROUTE_HOST}
    serving.kserve.io/s3-usehttps:          "1"
    serving.kserve.io/s3-verifyssl:         "0"
    serving.kserve.io/s3-region:            us-south
    serving.kserve.io/s3-useanoncredential: "false"
secrets:
- name: aws-connection-minio-data-connection
""")
print("Secret + ServiceAccount applied")

# ── 4. OVMS ServingRuntime ────────────────────────────────────────────────────
_apply(f"""
apiVersion: serving.kserve.io/v1alpha1
kind: ServingRuntime
metadata:
  name: ovms-runtime
  namespace: {NAMESPACE}
  labels:
    opendatahub.io/dashboard: "true"
  annotations:
    openshift.io/display-name: OpenVINO Model Server
    opendatahub.io/apiProtocol: REST
    prometheus.io/port: "8888"
    prometheus.io/path: /metrics
spec:
  multiModel: false
  supportedModelFormats:
  - name: onnx
    version: "1"
    autoSelect: true
  protocolVersions: [v2]
  containers:
  - name: kserve-container
    image: {OVMS_IMAGE}
    args:
    - --model_name={MODEL_NAME}
    - --port=8001
    - --rest_port=8888
    - --model_path=/mnt/models
    - --file_system_poll_wait_seconds=0
    - --grpc_bind_address=0.0.0.0
    - --rest_bind_address=0.0.0.0
    - --target_device=AUTO
    - --metrics_enable
    ports:
    - {{containerPort: 8888, name: http}}
    - {{containerPort: 8001, name: grpc}}
    resources:
      requests: {{cpu: "1", memory: 2Gi}}
      limits:   {{cpu: "2", memory: 4Gi}}
""")
print("ServingRuntime ovms-runtime applied")

# ── 5. Patch inferenceservice-config (KServe agent CA bundle) ────────────────
# Patches the KServe global config in redhat-ods-applications so the agent
# automatically mounts kserve-logger-ca-bundle at /etc/tls/logger/service-ca.crt.
# This allows the agent to verify TrustyAI's TLS cert and log inferences.
# Requires cluster-admin access to redhat-ods-applications namespace.
print("\n── 5. Patching inferenceservice-config for CA bundle ──")
import json as _j
_logger_raw = subprocess.run(
    ["oc","get","configmap","inferenceservice-config",
     "-n","redhat-ods-applications","-o","jsonpath={.data.logger}"],
    capture_output=True, text=True).stdout.strip()

# Check if patch is already applied
_already_patched = False
if _logger_raw:
    try:
        _already_patched = "kserve-logger-ca-bundle" in _j.loads(_logger_raw).get("caBundle","")
    except Exception:
        _already_patched = "kserve-logger-ca-bundle" in _logger_raw

if _already_patched:
    print(f"  ✅ inferenceservice-config already configured with CA bundle")
else:
    _agent_img = subprocess.run(
        ["oc","get","configmap","inferenceservice-config",
         "-n","redhat-ods-applications","-o","jsonpath={.data.agent}"],
        capture_output=True, text=True).stdout.strip()
    if _agent_img:
        try:
            _image = _j.loads(_agent_img).get("image","")
        except Exception:
            _image = ""
        if _image:
            _logger_val = _j.dumps({
                "image": _image, "memoryRequest": "100Mi", "memoryLimit": "1Gi",
                "cpuRequest": "100m", "cpuLimit": "1", "defaultUrl": "http://default-broker",
                "caBundle": "kserve-logger-ca-bundle", "caCertFile": "service-ca.crt",
                "tlsSkipVerify": False
            })
            _patch = _j.dumps([{"op": "add", "path": "/data/logger", "value": _logger_val}])
            subprocess.run(
                ["oc","patch","configmap","inferenceservice-config","-n","redhat-ods-applications",
                 "--type","merge","-p",'{"metadata":{"annotations":{"opendatahub.io/managed":"false"}}}'],
                capture_output=True, text=True)
            _r = subprocess.run(
                ["oc","patch","configmap","inferenceservice-config",
                 "-n","redhat-ods-applications","--type","json","-p",_patch],
                capture_output=True, text=True)
            if _r.returncode == 0:
                print(f"  ✅ inferenceservice-config patched with CA bundle config")
            else:
                print(f"  ⚠️  Could not patch (needs cluster-admin) — run patch-kserve.py once as cluster-admin")
        else:
            print("  ⚠️  Could not read agent image — run patch-kserve.py once as cluster-admin")
    else:
        print("  ⚠️  inferenceservice-config not found in redhat-ods-applications")

print("\n✅ Cell 1 complete → Cell 2 → Cell 3 → Cell 4 → Cell 5 → Cell 6 → Cell 7")


> **📋 Demo Note — Cell 2 (pre-run)**
>
> Standard Python dependencies for ML — scikit-learn, ONNX, KFP. Skip past this during the live demo.

In [2]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 2 — Install Dependencies
# ══════════════════════════════════════════════════════════════════════════
!pip install scikit-learn pandas numpy boto3 skl2onnx onnxruntime matplotlib seaborn requests \
    "kfp>=1.8.20,<2.0" "protobuf>=3.13.0,<4.0" -q


> **📋 Demo Note — Cell 3 (pre-run)**
>
> Configuration: derives cluster domain, builds endpoint URLs, acquires bearer token. Re-run after any kernel restart.

In [3]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 3 — Configuration
# Purpose : Re-run after kernel restart. Derives CLUSTER_DOMAIN, builds
#           all endpoint URLs (external + internal), acquires bearer token.
# ══════════════════════════════════════════════════════════════════════════
import subprocess, requests, warnings
warnings.filterwarnings("ignore")

NAMESPACE        = "telco-bias-demo"
MODEL_NAME       = "network-slice-model"
BUCKET_NAME      = "models"
MINIO_ENDPOINT   = f"http://minio.{NAMESPACE}.svc.cluster.local:9000"
MINIO_ACCESS_KEY = "THEACCESSKEY"
MINIO_SECRET_KEY = "THESECRETKEY"

# Reuse CLUSTER_DOMAIN if already set by Cell 1
try:
    CLUSTER_DOMAIN
except NameError:
    import re as _re
    _s = subprocess.run(["oc","whoami","--show-server"], capture_output=True, text=True)
    _m = _re.search(r"https?://api[.](.+?)(?:[:\d]+)?\s*$", _s.stdout.strip())
    CLUSTER_DOMAIN = "apps." + _m.group(1) if _m else ""
    if not CLUSTER_DOMAIN:
        _r2 = subprocess.run(["oc","get","ingress.config","cluster",
                               "-o","jsonpath={.spec.domain}"], capture_output=True, text=True)
        CLUSTER_DOMAIN = _r2.stdout.strip()
    if not CLUSTER_DOMAIN:
        raise RuntimeError("Set CLUSTER_DOMAIN manually: CLUSTER_DOMAIN = 'apps.YOUR-DOMAIN'")

# Guard against double apps. prefix
if CLUSTER_DOMAIN.startswith("apps.apps."):
    CLUSTER_DOMAIN = CLUSTER_DOMAIN[5:]
if not CLUSTER_DOMAIN.startswith("apps."):
    CLUSTER_DOMAIN = "apps." + CLUSTER_DOMAIN

_base = CLUSTER_DOMAIN
# External routes (OpenShift ingress)
INFER_URL      = f"https://{MODEL_NAME}-predictor-{NAMESPACE}.{_base}/v2/models/{MODEL_NAME}/infer"
TRUSTYAI_ROUTE = f"https://trustyai-service-{NAMESPACE}.{_base}"
KFP_ENDPOINT   = f"https://ds-pipeline-dspa-{NAMESPACE}.{_base}"
# Internal cluster URLs (used for warm-up inferences — bypasses ingress)
INFER_URL_INT  = f"http://{MODEL_NAME}-predictor.{NAMESPACE}.svc.cluster.local:8888/v2/models/{MODEL_NAME}/infer"
TRUSTYAI_INT   = f"http://trustyai-service.{NAMESPACE}.svc.cluster.local:80"

TOKEN   = subprocess.check_output(["oc","whoami","-t"], stderr=subprocess.DEVNULL).decode().strip()
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

print(f"Namespace    : {NAMESPACE}")
print(f"Model        : {MODEL_NAME}")
print(f"Infer (ext)  : {INFER_URL}")
print(f"Infer (int)  : {INFER_URL_INT}")
print(f"TrustyAI(ext): {TRUSTYAI_ROUTE}")
print(f"TrustyAI(int): {TRUSTYAI_INT}")


Namespace    : telco-bias-demo
Model        : network-slice-model
Infer (ext)  : https://network-slice-model-predictor-telco-bias-demo.apps.cluster-ts9lr.ts9lr.sandbox2570.opentlc.com/v2/models/network-slice-model/infer
Infer (int)  : http://network-slice-model-predictor.telco-bias-demo.svc.cluster.local:8888/v2/models/network-slice-model/infer
TrustyAI(ext): https://trustyai-service-telco-bias-demo.apps.cluster-ts9lr.ts9lr.sandbox2570.opentlc.com
TrustyAI(int): http://trustyai-service.telco-bias-demo.svc.cluster.local:80


> **📋 Demo Note — Cell 4 (pre-run)**
>
> "We generated a synthetic 5G network-slice dataset — 7,000 rows with features like Monthly Charges, Region, and Income. Importantly, the data has a **geographic bias baked in** — urban customers are over-represented in premium tiers."

In [4]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 4 — Generate Dataset
# Purpose : Synthesise 7 000-row telco network-slice dataset with slice_type,
#           latency_ms, throughput_mbps, region. Introduces demographic
#           correlation to create measurable SPD bias.
# ══════════════════════════════════════════════════════════════════════════
import pandas as pd, numpy as np
np.random.seed(42); n = 7000
df = pd.DataFrame({
    "tenure": np.random.randint(1,72,n), "MonthlyCharges": np.random.uniform(20,120,n),
    "TotalCharges": np.random.uniform(100,8000,n), "ContractType": np.random.choice([0,1,2],n),
    "PaymentMethod": np.random.choice([0,1,2,3],n),
    "Region": np.random.choice(["Urban","Suburban","Rural"],n,p=[0.5,0.3,0.2]),
    "IncomeProxy": np.random.choice(["High","Medium","Low"],n,p=[0.3,0.4,0.3])
})
def assign_tier(row):
    score = row["MonthlyCharges"]/120.0
    if row["Region"]=="Urban" and row["IncomeProxy"]=="High": score += 0.35
    elif row["Region"]=="Rural": score -= 0.25
    score += np.random.normal(0,0.1)
    return 2 if score>0.65 else (1 if score>0.35 else 0)
df["NetworkSliceTier"] = df.apply(assign_tier, axis=1)
df["Region_enc"] = df["Region"].map({"Urban":0,"Suburban":1,"Rural":2})
df["Income_enc"] = df["IncomeProxy"].map({"High":0,"Medium":1,"Low":2})
df.to_csv("/tmp/network_slice_data.csv", index=False)
print(pd.crosstab(df["Region"], df["NetworkSliceTier"], normalize="index").round(3))

NetworkSliceTier      0      1      2
Region                               
Rural             0.512  0.357  0.131
Suburban          0.218  0.357  0.425
Urban             0.157  0.303  0.541


> **📋 Demo Note — Cell 5 (pre-run)**
>
> "We trained a neural network on this biased data and exported it as ONNX. The model learned the geographic bias from the training labels — it'll score urban customers higher than rural ones for the same profile."

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 5 — Train MLP, Export ONNX & Strip ZipMap
# Purpose : Train MLP on generated dataset, export ONNX (opset 12),
#           strip ZipMap, replace with ArgMax → Cast(INT64→FP32).
#           TrustyAI requires FP32 output for payload reconciliation.
#           Output: /tmp/network_slice_model.onnx
# ══════════════════════════════════════════════════════════════════════════
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnx
from onnx import helper, TensorProto

FEATURES = ["tenure","MonthlyCharges","TotalCharges","ContractType",
            "PaymentMethod","Region_enc","Income_enc"]
X = df[FEATURES].astype(float)
y = df["NetworkSliceTier"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
clf.fit(X_train, y_train)
print(f"Accuracy: {clf.score(X_test, y_test):.3f}")

onnx_m = convert_sklearn(clf,
    initial_types=[("float_input", FloatTensorType([None, 7]))],
    target_opset=12)
with open("/tmp/network_slice_model.onnx", "wb") as f:
    f.write(onnx_m.SerializeToString())
print("ONNX exported")

# ── Strip ZipMap, add ArgMax + Cast(INT64 → FP32) ────────────────────────────
# TrustyAI's KServe payload reconciler fails to write labeled records when
# the model output type is INT64. Casting to FP32 fixes reconciliation.
m = onnx.load("/tmp/network_slice_model.onnx")
new_nodes, zipmap_input = [], None
for node in m.graph.node:
    if node.op_type == "ZipMap":
        zipmap_input = node.input[0]
    else:
        new_nodes.append(node)

if zipmap_input:
    # ArgMax: probabilities → INT64 class index
    argmax_node = helper.make_node(
        "ArgMax", inputs=[zipmap_input], outputs=["label_int64"], axis=1, keepdims=0)
    # Cast: INT64 → FP32 so TrustyAI reconciler handles it correctly
    cast_node = helper.make_node(
        "Cast", inputs=["label_int64"], outputs=["label"],
        to=TensorProto.FLOAT)
    new_nodes.extend([argmax_node, cast_node])
    new_out = [helper.make_tensor_value_info("label", TensorProto.FLOAT, None)]
    g = helper.make_graph(new_nodes, m.graph.name,
                          list(m.graph.input), new_out,
                          list(m.graph.initializer))
    m2 = helper.make_model(g, opset_imports=m.opset_import)
    m2.ir_version = m.ir_version
    onnx.save(m2, "/tmp/network_slice_model.onnx")
    print("ZipMap stripped, ArgMax + Cast(FP32) added ✅")
else:
    print("No ZipMap found (already clean)")

ops = set(n.op_type for n in onnx.load("/tmp/network_slice_model.onnx").graph.node)
print(f"Final ops: {ops}")
assert "ZipMap" not in ops
print("✅ Model ready — run Cell 6 to upload to MinIO")


> **📋 Demo Note — Cell 6 (pre-run)**
>
> "The ONNX model is uploaded to MinIO and served live via OpenShift AI's KServe in RawDeployment mode — no Istio, no Knative required."

In [6]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 6 — Upload Model to MinIO S3
# Purpose : Upload ONNX to s3://models/network-slice-model/1/model.onnx
#           (path OVMS storage-initializer expects). Verify upload.
# ══════════════════════════════════════════════════════════════════════════
import boto3
from botocore.client import Config

s3 = boto3.client("s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version="s3v4"),
    verify=False)

try:
    s3.create_bucket(Bucket=BUCKET_NAME)
    print(f"Bucket '{BUCKET_NAME}' created")
except Exception:
    print(f"Bucket '{BUCKET_NAME}' already exists (ok)")

s3_key = f"{MODEL_NAME}/1/model.onnx"
s3.upload_file("/tmp/network_slice_model.onnx", BUCKET_NAME, s3_key)
print(f"✅ Uploaded → s3://{BUCKET_NAME}/{s3_key}")

objs = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=MODEL_NAME)
for o in objs.get("Contents", []):
    print(f"  {o['Key']}  {o['Size']/1024:.1f} KB")


Bucket 'models' created
✅ Uploaded → s3://models/network-slice-model/1/model.onnx
  network-slice-model/1/model.onnx  11.7 KB


> **📋 Demo Note — Cell 7** *(~30 seconds)*
>
> "Here TrustyAI is monitoring our model in real time. We seeded 200 inferences — these establish what 'normal' looks like. TrustyAI automatically registers the model and starts tracking every prediction."

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 7 — Deploy TrustyAI → InferenceService → Seed Inferences (RHOAI 2.x)
# ══════════════════════════════════════════════════════════════════════════
#
# DEPLOYMENT ORDER (critical for TrustyAI logging to work):
#   1. Create kserve-logger-ca-bundle ConfigMap
#   2. Deploy TrustyAI FIRST — wait for Running
#   3. Deploy InferenceService — TrustyAI operator detects it and sets up
#      the logger volume mount on the predictor pod automatically
#   4. Seed inferences via oc exec → agent → TrustyAI
#
# RHOAI 2.x: No OSSM/Istio required. RawDeployment mode.
# ══════════════════════════════════════════════════════════════════════════
import subprocess, time, requests, json, warnings, tempfile, os
warnings.filterwarnings("ignore")

TOKEN   = subprocess.check_output(["oc","whoami","-t"], stderr=subprocess.DEVNULL).decode().strip()
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
print(f"Bearer token: {TOKEN[:8]}…")

FEATURES = ["tenure","MonthlyCharges","TotalCharges","ContractType",
            "PaymentMethod","Region_enc","Income_enc"]

# ── 1. Create kserve-logger-ca-bundle FIRST ───────────────────────────────────
# Must exist before ISVC is created so TrustyAI operator can mount it.
print("── 1. kserve-logger-ca-bundle ──")
_ca_exists = subprocess.run(
    ["oc","get","configmap","kserve-logger-ca-bundle","-n",NAMESPACE],
    capture_output=True, text=True).returncode == 0
if not _ca_exists:
    _ca_data = subprocess.check_output(
        ["oc","get","configmap","openshift-service-ca.crt","-n",NAMESPACE,
         "-o","jsonpath={.data.service-ca\\.crt}"], text=True)
    _tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".crt", mode="w")
    _tmp.write(_ca_data); _tmp.close()
    subprocess.run(
        ["oc","create","configmap","kserve-logger-ca-bundle",
         f"--from-file=service-ca.crt={_tmp.name}","-n",NAMESPACE],
        capture_output=True, text=True)
    os.unlink(_tmp.name)
    print("  ✅ kserve-logger-ca-bundle created")
else:
    print("  kserve-logger-ca-bundle already exists")

# ── 2. Deploy TrustyAI FIRST ─────────────────────────────────────────────────
print("\n── 2. Deploying TrustyAI ──")
_tai_exists = subprocess.run(
    ["oc","get","trustyaiservice","trustyai-service","-n",NAMESPACE],
    capture_output=True, text=True).returncode == 0
if not _tai_exists:
    _apply(f"""
apiVersion: trustyai.opendatahub.io/v1alpha1
kind: TrustyAIService
metadata:
  name: trustyai-service
  namespace: {NAMESPACE}
spec:
  storage:
    format: PVC
    folder: /data
    size: 1Gi
  data:
    filename: data.csv
    format: CSV
  metrics:
    schedule: "5s"
    batchSize: 5000
""")
else:
    print("  TrustyAIService already exists")

# ── 3. Wait for TrustyAI Running ─────────────────────────────────────────────
print("\n── 3. Waiting for TrustyAI pod Running (≤300s) ──")
for _i in range(60):
    _phase = subprocess.run(
        ["oc","get","pod","-n",NAMESPACE,"-l","app=trustyai-service",
         "-o","jsonpath={.items[0].status.phase}"],
        capture_output=True, text=True).stdout.strip()
    if _phase == "Running":
        print("  ✅ TrustyAI pod Running"); break
    if _i % 4 == 0:
        print(f"  [{_i*5}s] phase={_phase!r}")
    time.sleep(5)
else:
    raise RuntimeError("TrustyAI not Running after 300s")

# ── 4. Deploy InferenceService (AFTER TrustyAI is Running) ───────────────────
# TrustyAI operator will detect this ISVC and properly set up logger volume mounts.
print("\n── 4. Deploying InferenceService ──")
_isvc_exists = subprocess.run(
    ["oc","get","inferenceservice",MODEL_NAME,"-n",NAMESPACE],
    capture_output=True, text=True).returncode == 0
if not _isvc_exists:
    _apply(f"""
apiVersion: serving.kserve.io/v1beta1
kind: InferenceService
metadata:
  name: {MODEL_NAME}
  namespace: {NAMESPACE}
  labels:
    opendatahub.io/dashboard: "true"
    trustyai.opendatahub.io/monitoring: "true"
  annotations:
    openshift.io/display-name: {MODEL_NAME}
    serving.kserve.io/deploymentMode: RawDeployment
spec:
  predictor:
    serviceAccountName: kserve-minio-sa
    minReplicas: 1
    model:
      modelFormat: {{name: onnx, version: "1"}}
      runtime: ovms-runtime
      storageUri: s3://{BUCKET_NAME}/{MODEL_NAME}
      resources:
        requests: {{cpu: "1", memory: 4Gi}}
        limits:   {{cpu: "2", memory: 8Gi}}
""")
else:
    print(f"  InferenceService '{MODEL_NAME}' already exists")

# ── 5. Wait for ISVC Loaded ───────────────────────────────────────────────────
print("\n── 5. Waiting for ISVC → Loaded (timeout=300s) ──")
_loaded = False
for _i in range(60):
    _val = subprocess.run(
        ["oc","get","inferenceservice",MODEL_NAME,"-n",NAMESPACE,"-o",
         "jsonpath={.status.modelStatus.states.activeModelState}"],
        capture_output=True, text=True).stdout.strip()
    if _i % 6 == 0:
        _pod = subprocess.run(
            ["oc","get","pod","-n",NAMESPACE,
             "-l",f"serving.kserve.io/inferenceservice={MODEL_NAME}",
             "--no-headers","-o","custom-columns=NAME:.metadata.name,STATUS:.status.phase"],
            capture_output=True, text=True).stdout.strip()
        print(f"  [{_i*5}s] ISVC={_val!r}  {_pod}")
    if _val == "Loaded":
        print("  ✅ ISVC Loaded"); _loaded = True; break
    time.sleep(5)
if not _loaded:
    raise RuntimeError("ISVC not Loaded after 300s")

# Wait for TrustyAI to patch ISVC and pod to restart with logger volume mount
print("  Waiting 30s for TrustyAI to patch predictor pod…")
time.sleep(30)

# Verify CA bundle is mounted
_pred_pod = subprocess.run(
    ["oc","get","pod","-n",NAMESPACE,
     "-l",f"serving.kserve.io/inferenceservice={MODEL_NAME}",
     "-o","jsonpath={.items[0].metadata.name}"],
    capture_output=True, text=True).stdout.strip()
_ca_check = subprocess.run(
    ["oc","exec","-n",NAMESPACE,_pred_pod,"-c","agent","--",
     "ls","/etc/tls/logger/service-ca.crt"],
    capture_output=True, text=True)
if _ca_check.returncode == 0:
    print("  ✅ CA bundle mounted at /etc/tls/logger/service-ca.crt")
else:
    print("  ⚠️  CA bundle not yet mounted — TrustyAI may need more time to patch")
    time.sleep(20)

# ── 6. Seed inferences via oc exec (agent → TrustyAI) ────────────────────────
print(f"\n── 6. Seeding 200 inferences via oc exec ──")
_pred_pod = subprocess.run(
    ["oc","get","pod","-n",NAMESPACE,
     "-l",f"serving.kserve.io/inferenceservice={MODEL_NAME}",
     "-o","jsonpath={.items[0].metadata.name}"],
    capture_output=True, text=True).stdout.strip()
print(f"  Predictor pod: {_pred_pod}")

_urban_rows = X_test[df.loc[X_test.index,"Region"]=="Urban"].head(75)
_rural_rows  = X_test[df.loc[X_test.index,"Region"]=="Rural"].head(75)
_seed_rows   = list(X_test.head(50).iterrows()) + \
               list(_urban_rows.iterrows()) + \
               list(_rural_rows.iterrows())

_sent = 0
for _, _row in _seed_rows:
    _data = json.dumps({"inputs":[{"name":"float_input","shape":[1,7],
                                   "datatype":"FP32",
                                   "data":[float(v) for v in _row[FEATURES].values]}]})
    _rc = subprocess.run(
        ["oc","exec","-n",NAMESPACE,_pred_pod,"-c","kserve-container","--",
         "curl","-s","--max-time","5","-X","POST",
         f"http://localhost:9081/v2/models/{MODEL_NAME}/infer",
         "-H","Content-Type: application/json","-d",_data,
         "-o","/dev/null","-w","%{http_code}"],
        capture_output=True, text=True)
    if _rc.stdout.strip() == "200":
        _sent += 1
    time.sleep(0.05)
print(f"  {_sent}/{len(_seed_rows)} inferences sent")

# ── 7. Wait for TrustyAI to register model ───────────────────────────────────
print("\n── 7. Waiting for TrustyAI to register model (≤90s) ──")
for _i in range(18):
    time.sleep(5)
    try:
        _resp = requests.get(f"{TRUSTYAI_INT}/info", headers=HEADERS,
                             verify=False, timeout=10)
        if _resp.status_code == 200 and MODEL_NAME in str(_resp.json()):
            _obs = _resp.json().get(MODEL_NAME,{}).get("data",{}).get("observations",0)
            if _obs > 0:
                print(f"  ✅ Model registered — observations: {_obs}")
                print(_resp.text[:400])
                break
        print(f"  [{_i*5}s] waiting…")
    except Exception as _e:
        print(f"  [{_i*5}s] {_e}")
else:
    print("⚠️  Model not yet registered. Check TrustyAI logs:")
    print(f"  oc logs -n {NAMESPACE} -l app=trustyai-service --tail=20")
    print(f"  oc logs -n {NAMESPACE} {_pred_pod} -c agent --tail=10")

print("\n✅ Cell 7 complete → Cell 8 (Register SPD/DIR monitors)")


> **📋 Demo Note — Cell 8** *(~30 seconds)*
>
> "We plug into OpenShift's built-in observability stack. Prometheus scrapes TrustyAI's metrics, Grafana visualizes them. Show the Grafana dashboard — you can see SPD and DIR trending in real time."

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 8 — Prometheus Monitoring & Grafana Dashboard (RHOAI 2.x)
# Purpose : Deploy Prometheus Pushgateway so notebooks can push live SPD/DIR
#           values directly to Prometheus. Grafana then shows real SPD/DIR
#           gauges that update as the demo progresses (baseline → bias → fix).
#
# Architecture:
#   Notebook cells compute SPD/DIR manually (proven accurate)
#         ↓  HTTP POST
#   Pushgateway (port 9091) in telco-bias-demo namespace
#         ↓  scraped every 15s
#   Prometheus (user workload monitoring)
#         ↓
#   Grafana dashboard — live SPD/DIR gauges
# ══════════════════════════════════════════════════════════════════════════
import subprocess, requests, warnings, json as _jmod, textwrap, time
warnings.filterwarnings("ignore")

def _apply_soft(yaml_str, path, label):
    with open(path, "w") as _f:
        _f.write(textwrap.dedent(yaml_str).strip() + "\n")
    _r = subprocess.run(["oc","apply","-f",path], capture_output=True, text=True)
    _out = (_r.stdout + _r.stderr).strip()
    if _r.returncode == 0:
        print(f"  ✅ {_out}")
    else:
        print(f"  ⚠️  {label}: {_out[:200]}")

# ── 1. Deploy Prometheus Pushgateway ─────────────────────────────────────────
print("── 1. Deploying Prometheus Pushgateway ──")
_apply_soft(f"""
apiVersion: apps/v1
kind: Deployment
metadata:
  name: pushgateway
  namespace: {NAMESPACE}
  labels:
    app: pushgateway
spec:
  replicas: 1
  selector:
    matchLabels:
      app: pushgateway
  template:
    metadata:
      labels:
        app: pushgateway
    spec:
      containers:
      - name: pushgateway
        image: prom/pushgateway:latest
        ports:
        - containerPort: 9091
          name: http
        resources:
          requests:
            cpu: 50m
            memory: 64Mi
          limits:
            cpu: 200m
            memory: 128Mi
---
apiVersion: v1
kind: Service
metadata:
  name: pushgateway
  namespace: {NAMESPACE}
  labels:
    app: pushgateway
spec:
  ports:
  - port: 9091
    targetPort: 9091
    name: http
  selector:
    app: pushgateway
""", "/tmp/pushgateway.yaml", "Pushgateway")

# Wait for Pushgateway to be ready
print("  Waiting for Pushgateway pod...")
for _i in range(24):
    _phase = subprocess.run(
        ["oc","get","pod","-n",NAMESPACE,"-l","app=pushgateway",
         "-o","jsonpath={.items[0].status.phase}"],
        capture_output=True, text=True).stdout.strip()
    if _phase == "Running":
        print(f"  ✅ Pushgateway Running"); break
    time.sleep(5)
else:
    print("  ⚠️  Pushgateway not ready after 2m")

# ── 2. ServiceMonitor for Pushgateway ────────────────────────────────────────
print("\n── 2. Applying Pushgateway ServiceMonitor ──")
_apply_soft(f"""
apiVersion: monitoring.coreos.com/v1
kind: ServiceMonitor
metadata:
  name: pushgateway-monitor
  namespace: {NAMESPACE}
  labels:
    app: pushgateway
spec:
  selector:
    matchLabels:
      app: pushgateway
  endpoints:
  - port: http
    interval: 15s
    honorLabels: true
""", "/tmp/pushgateway-sm.yaml", "Pushgateway ServiceMonitor")

# ── 3. Existing TrustyAI ServiceMonitor + PrometheusRule ─────────────────────
print("\n── 3. TrustyAI ServiceMonitor + PrometheusRule ──")
_apply_soft(f"""
apiVersion: monitoring.coreos.com/v1
kind: ServiceMonitor
metadata:
  name: trustyai-bias-monitor
  namespace: {NAMESPACE}
  labels:
    app: trustyai-service
spec:
  selector:
    matchLabels:
      app: trustyai-service
  endpoints:
  - port: http
    path: /q/metrics
    interval: 30s
""", "/tmp/trustyai-sm.yaml", "TrustyAI ServiceMonitor")

_apply_soft(f"""
apiVersion: monitoring.coreos.com/v1
kind: PrometheusRule
metadata:
  name: trustyai-bias-alerts
  namespace: {NAMESPACE}
  labels:
    openshift.io/prometheus-rule-evaluation-scope: leaf-prometheus
spec:
  groups:
  - name: trustyai.bias
    rules:
    - alert: NetworkSliceBiasDetected
      expr: abs(trustyai_spd_live{{model="{MODEL_NAME}"}}) > 0.15
      for: 1m
      labels:
        severity: critical
      annotations:
        summary: "Geographic bias detected in 5G slice allocation"
        description: "SPD={{{{ $value | humanize }}}} exceeds ±0.15 in {MODEL_NAME}."
    - alert: DIRBelowThreshold
      expr: trustyai_dir_live{{model="{MODEL_NAME}"}} < 0.80
      for: 1m
      labels:
        severity: warning
      annotations:
        summary: "Disparate Impact Ratio below regulatory threshold"
        description: "DIR={{{{ $value | humanize }}}} below 0.80 in {MODEL_NAME}."
""", "/tmp/trustyai-rule.yaml", "PrometheusRule")

# ── 4. Enable user workload monitoring ────────────────────────────────────────
print("\n── 4. Enabling user workload monitoring ──")
_uwm = subprocess.run(["oc","apply","-f","-"], input="""apiVersion: v1
kind: ConfigMap
metadata:
  name: cluster-monitoring-config
  namespace: openshift-monitoring
data:
  config.yaml: |
    enableUserWorkload: true
""", capture_output=True, text=True)
print(f"  {'✅' if _uwm.returncode==0 else '⚠️ '} {(_uwm.stdout+_uwm.stderr).strip()[:100]}")

# ── 5. Grafana dashboard — live SPD/DIR from Pushgateway ─────────────────────
print("\n── 5. Deploying Grafana dashboard with live SPD/DIR gauges ──")

PUSHGATEWAY_URL = f"http://pushgateway.{NAMESPACE}.svc.cluster.local:9091"

_dashboard = {
    "title": "TrustyAI - Bias Monitor (Network Slice)",
    "uid": "trustyai-bias-005",
    "schemaVersion": 38,
    "refresh": "15s",
    "time": {"from": "now-1h", "to": "now"},
    "panels": [
        {
            "id": 1,
            "title": "Inferences Monitored by TrustyAI",
            "description": "Total observations logged — grows as demo progresses",
            "type": "gauge",
            "datasource": {"type": "prometheus", "uid": "prometheus"},
            "targets": [{"expr": f'trustyai_model_observations_total{{model="{MODEL_NAME}"}}',
                         "legendFormat": "Count"}],
            "options": {"reduceOptions": {"calcs": ["lastNotNull"]}},
            "fieldConfig": {"defaults": {
                "min": 0, "max": 5000,
                "thresholds": {"mode": "absolute", "steps": [{"color": "blue", "value": None}]}
            }},
            "gridPos": {"h": 8, "w": 8, "x": 0, "y": 0}
        },
        {
            "id": 2,
            "title": "SPD — Statistical Parity Difference (LIVE)",
            "description": "Urban vs Rural Premium allocation gap. Green=fair (±0.10), Red=bias (>±0.15). Updates when notebook cells run.",
            "type": "gauge",
            "datasource": {"type": "prometheus", "uid": "prometheus"},
            "targets": [{"expr": f'trustyai_spd_live{{model="{MODEL_NAME}"}}',
                         "legendFormat": "SPD"}],
            "options": {"reduceOptions": {"calcs": ["lastNotNull"]}},
            "fieldConfig": {"defaults": {
                "min": -0.5, "max": 0.5, "decimals": 3,
                "thresholds": {"mode": "absolute", "steps": [
                    {"color": "red",    "value": None},
                    {"color": "green",  "value": -0.10},
                    {"color": "yellow", "value": 0.10},
                    {"color": "red",    "value": 0.15}
                ]}
            }},
            "gridPos": {"h": 8, "w": 8, "x": 8, "y": 0}
        },
        {
            "id": 3,
            "title": "DIR — Disparate Impact Ratio (LIVE)",
            "description": "P(Premium|Rural)/P(Premium|Urban). Green=compliant (≥0.90), Red=violation (<0.80). Updates when notebook cells run.",
            "type": "gauge",
            "datasource": {"type": "prometheus", "uid": "prometheus"},
            "targets": [{"expr": f'trustyai_dir_live{{model="{MODEL_NAME}"}}',
                         "legendFormat": "DIR"}],
            "options": {"reduceOptions": {"calcs": ["lastNotNull"]}},
            "fieldConfig": {"defaults": {
                "min": 0, "max": 2.0, "decimals": 3,
                "thresholds": {"mode": "absolute", "steps": [
                    {"color": "red",    "value": None},
                    {"color": "yellow", "value": 0.80},
                    {"color": "green",  "value": 0.90}
                ]}
            }},
            "gridPos": {"h": 8, "w": 8, "x": 16, "y": 0}
        }
    ]
}

_db_json = _jmod.dumps(_dashboard, indent=2)
_db_json_indented = "\n".join("    " + l for l in _db_json.splitlines())
_cm_yaml = (
    "apiVersion: v1\nkind: ConfigMap\nmetadata:\n"
    "  name: trustyai-bias-dashboard\n  namespace: openshift-config-managed\n"
    "  labels:\n    console.openshift.io/dashboard: \"true\"\n"
    "data:\n  trustyai-bias-dashboard.json: |\n" + _db_json_indented + "\n"
)
with open("/tmp/trustyai-bias-dashboard-cm.yaml","w") as _f: _f.write(_cm_yaml)
_rc = subprocess.run(["oc","apply","-f","/tmp/trustyai-bias-dashboard-cm.yaml"],
                     capture_output=True, text=True)
print(f"  {'✅' if _rc.returncode==0 else '⚠️ '} {_rc.stdout.strip()}")

# ── 6. Helper function (available to all subsequent cells) ────────────────────
def push_spd_dir(spd_val, dir_val, step="current"):
    """Push SPD and DIR values to Prometheus via Pushgateway."""
    payload = (
        f'# HELP trustyai_spd_live SPD computed manually from model predictions\n'
        f'# TYPE trustyai_spd_live gauge\n'
        f'trustyai_spd_live{{model="{MODEL_NAME}",step="{step}",protected="Region"}} {spd_val}\n'
        f'# HELP trustyai_dir_live DIR computed manually from model predictions\n'
        f'# TYPE trustyai_dir_live gauge\n'
        f'trustyai_dir_live{{model="{MODEL_NAME}",step="{step}",protected="Region"}} {dir_val}\n'
    )
    try:
        r = requests.post(
            f"{PUSHGATEWAY_URL}/metrics/job/bias-monitor/model/{MODEL_NAME}",
            data=payload, headers={"Content-Type": "text/plain"}, timeout=10
        )
        return r.status_code == 200
    except Exception as e:
        print(f"  ⚠️  Pushgateway push failed: {e}")
        return False

print("\n✅ push_spd_dir() helper loaded — available in Cells 8, 10, 12")

# ── 7. Access info ────────────────────────────────────────────────────────────
_console_host = subprocess.run(
    ["oc","get","route","console","-n","openshift-console",
     "-o","jsonpath={.spec.host}"], capture_output=True, text=True).stdout.strip()
print(f"""
╔══════════════════════════════════════════════════════════════════════════╗
║  OPEN GRAFANA: Observe > Dashboards > TrustyAI - Bias Monitor          ║
║  https://{_console_host}
╠══════════════════════════════════════════════════════════════════════════╣
║  DASHBOARD PANELS (live values pushed from notebook):                  ║
║  [Inferences Monitored]  [SPD Gauge — LIVE]  [DIR Gauge — LIVE]       ║
╠══════════════════════════════════════════════════════════════════════════╣
║  DEMO FLOW:                                                             ║
║  Cell 8  → baseline SPD pushed → Grafana shows ~0.03 (green)           ║
║  Cell 9  → bias injected                                               ║
║  Cell 10 → biased SPD pushed  → Grafana spikes to ~0.25 (red)          ║
║  Cell 11b→ fair retrain                                                ║
║  Cell 12 → fair SPD pushed    → Grafana drops to ~0.05 (green)         ║
╚══════════════════════════════════════════════════════════════════════════╝
""")
print("✅ Cell 8 complete → run Cell 9 to register SPD/DIR and push baseline to Grafana")


> **📋 Demo Note — Cell 8b** *(cluster-admin step, pre-run)*
>
> Applies user workload monitoring config and the Grafana dashboard ConfigMap. Run once as part of setup — not during the live demo.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 8b — Apply Monitoring Config & Dashboard (Cluster-Admin Required)
# Purpose : Apply user-workload-monitoring ConfigMap and Grafana dashboard.
#           Steps 1–2 require cluster-admin access (run pre-req #5 in Cell 1).
#           Step 3 shows Prometheus queries for Observe > Metrics.
# ══════════════════════════════════════════════════════════════════════════
import subprocess, json as _jmod, warnings
warnings.filterwarnings("ignore")

# ── 1. User Workload Monitoring ConfigMap ─────────────────────────────────────
print("── 1. User Workload Monitoring ──")
_uwm_yaml = """apiVersion: v1
kind: ConfigMap
metadata:
  name: cluster-monitoring-config
  namespace: openshift-monitoring
data:
  config.yaml: |
    enableUserWorkload: true
"""
with open("/tmp/user-workload-monitoring.yaml", "w") as _f: _f.write(_uwm_yaml)
_r = subprocess.run(["oc","apply","-f","/tmp/user-workload-monitoring.yaml"], capture_output=True, text=True)
if _r.returncode == 0:
    print(f"  ✅ {_r.stdout.strip()}")
else:
    print(f"  ⚠️  Needs cluster-admin pre-req #5 from Cell 1")
    print(f"     oc apply -f /tmp/user-workload-monitoring.yaml")

# ── 2. Grafana Dashboard ConfigMap ────────────────────────────────────────────
# Panel order: 1) Inferences  2) SPD  3) DIR
print("\n── 2. Grafana Bias Dashboard ──")
_dashboard = {
    "title": "TrustyAI - Bias Monitor (Network Slice)",
    "uid": "trustyai-bias-005",
    "schemaVersion": 38,
    "refresh": "15s",
    "time": {"from": "now-3h", "to": "now"},
    "panels": [
        {
            "id": 1,
            "title": "Inferences Monitored by TrustyAI",
            "description": "Total observations TrustyAI has logged from the model",
            "type": "graph",
            "datasource": {"type": "prometheus", "uid": "prometheus"},
            "targets": [{"expr": f'sum(trustyai_model_observations_total{{model="{MODEL_NAME}"}}) by (model)',
                         "legendFormat": "Observations"}],
            "lines": True, "fill": 2, "linewidth": 3, "nullPointMode": "null",
            "legend": {"show": True, "current": True, "values": True, "max": True, "alignAsTable": False},
            "yaxes": [{"format": "short", "label": "Count", "show": True, "min": 0},
                      {"format": "short", "show": False}],
            "xaxis": {"mode": "time", "show": True},
            "thresholds": [],
            "gridPos": {"h": 8, "w": 24, "x": 0, "y": 0}
        },
        {
            "id": 2,
            "title": "SPD \u2014 Statistical Parity Difference  |  Fair: -0.10 to +0.10  |  Alert: > \u00b10.15",
            "description": "Urban vs Rural Premium allocation gap. Baseline \u2192 Bias (Cell 10) \u2192 Retrain (Cell 13).",
            "type": "graph",
            "datasource": {"type": "prometheus", "uid": "prometheus"},
            "targets": [{"expr": f'trustyai_spd_live{{model="{MODEL_NAME}"}}',
                         "legendFormat": "SPD ({{step}})"}],
            "lines": True, "fill": 2, "linewidth": 3, "nullPointMode": "null",
            "legend": {"show": True, "current": True, "values": True, "avg": True,
                       "max": True, "min": True, "alignAsTable": False},
            "yaxes": [{"format": "short", "label": "SPD", "show": True, "min": -0.5, "max": 0.5},
                      {"format": "short", "show": False}],
            "xaxis": {"mode": "time", "show": True},
            "thresholds": [
                {"colorMode": "warning",  "op": "gt", "value":  0.10, "fill": True, "line": True, "yaxis": "left"},
                {"colorMode": "critical", "op": "gt", "value":  0.15, "fill": True, "line": True, "yaxis": "left"},
            ],
            "gridPos": {"h": 10, "w": 24, "x": 0, "y": 8}
        },
        {
            "id": 3,
            "title": "DIR \u2014 Disparate Impact Ratio  |  Fair: 0.80\u20131.20  |  Alert: < 0.80",
            "description": "P(Premium|Rural)/P(Premium|Urban). Below 0.80 = regulatory violation.",
            "type": "graph",
            "datasource": {"type": "prometheus", "uid": "prometheus"},
            "targets": [{"expr": f'trustyai_dir_live{{model="{MODEL_NAME}"}}',
                         "legendFormat": "DIR ({{step}})"}],
            "lines": True, "fill": 2, "linewidth": 3, "nullPointMode": "null",
            "legend": {"show": True, "current": True, "values": True, "avg": True,
                       "max": True, "min": True, "alignAsTable": False},
            "yaxes": [{"format": "short", "label": "DIR", "show": True, "min": 0, "max": 2.0},
                      {"format": "short", "show": False}],
            "xaxis": {"mode": "time", "show": True},
            "thresholds": [
                {"colorMode": "critical", "op": "lt", "value": 0.80, "fill": True, "line": True, "yaxis": "left"},
            ],
            "gridPos": {"h": 10, "w": 24, "x": 0, "y": 18}
        }
    ]
}

_db_json = _jmod.dumps(_dashboard, indent=2)
_db_indented = "\n".join("    " + l for l in _db_json.splitlines())
_db_yaml = (
    "apiVersion: v1\nkind: ConfigMap\nmetadata:\n"
    "  name: trustyai-bias-dashboard\n  namespace: openshift-config-managed\n"
    "  labels:\n    console.openshift.io/dashboard: \"true\"\n"
    "data:\n  trustyai-bias-dashboard.json: |\n" + _db_indented + "\n"
)
with open("/tmp/grafana-bias-dashboard.yaml", "w") as _f: _f.write(_db_yaml)
_r2 = subprocess.run(["oc","apply","-f","/tmp/grafana-bias-dashboard.yaml"], capture_output=True, text=True)
if _r2.returncode == 0:
    print(f"  ✅ {_r2.stdout.strip()}")
else:
    print(f"  ⚠️  Needs cluster-admin pre-req #5 from Cell 1")
    print(f"     oc apply -f /tmp/grafana-bias-dashboard.yaml")

# ── 3. Prometheus queries for Observe > Metrics ───────────────────────────────
_console_host = subprocess.run(
    ["oc","get","route","console","-n","openshift-console",
     "-o","jsonpath={.spec.host}"],
    capture_output=True, text=True).stdout.strip()

print(f"""
── 3. Prometheus Queries for Observe > Metrics ──
╔═══════════════════════════════════════════════════════════════════════════╗
║  https://{_console_host}
║  → Observe → Metrics → paste query → Run queries                         ║
╠═══════════════════════════════════════════════════════════════════════════╣
║  Inferences:                                                              ║
║    sum(trustyai_model_observations_total{{model="{MODEL_NAME}"}}) by (model)
║  SPD (all steps — baseline/biased/fair):                                 ║
║    trustyai_spd_live{{model="{MODEL_NAME}"}}
║  DIR (all steps):                                                         ║
║    trustyai_dir_live{{model="{MODEL_NAME}"}}
╠═══════════════════════════════════════════════════════════════════════════╣
║  Grafana Dashboard:                                                       ║
║  Observe → Dashboards → TrustyAI - Bias Monitor (Network Slice)          ║
║  Panel order: [Inferences] → [SPD] → [DIR]                               ║
╚═══════════════════════════════════════════════════════════════════════════╝
""")
print("✅ Cell 8b complete → run Cell 9 (Register SPD/DIR + push baseline to Grafana)")


> **📋 Demo Note — Cell 9 + Cell 10 — THE WOW MOMENT** *(~1 minute)*
>
> "Now we register SPD — Statistical Parity Difference — to measure if Region is affecting outcomes unfairly. Then we simulate what happens in production: rural customers are charged low rates, urban customers high."
>
> **Watch the SPD spike** — TrustyAI fires an alert the moment it crosses the threshold. No manual inspection, no waiting for a quarterly audit."

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 9 — Register SPD & DIR Bias Monitors via TrustyAI (RHOAI 2.x)
# Purpose : Apply nameMapping for readable column names, then register
#           scheduled SPD and DIR monitors using the /request endpoint.
#           The /request endpoint creates background scheduled jobs that
#           compute fairness metrics on every schedule tick (5s) and
#           emit trustyai_spd / trustyai_dir to /q/metrics for Prometheus.
# ══════════════════════════════════════════════════════════════════════════
import requests, subprocess, time

TOKEN   = subprocess.check_output(["oc","whoami","-t"], stderr=subprocess.DEVNULL).decode().strip()
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

# ── 1. Apply nameMapping (human-readable column names) ───────────────────────
nm = requests.post(f"{TRUSTYAI_INT}/info/names", headers=HEADERS, timeout=30, json={
    "modelId": MODEL_NAME,
    "inputMapping": {
        "float_input-0": "tenure",
        "float_input-1": "MonthlyCharges",
        "float_input-2": "TotalCharges",
        "float_input-3": "ContractType",
        "float_input-4": "PaymentMethod",
        "float_input-5": "Region",
        "float_input-6": "IncomeProxy"
    },
    "outputMapping": {"label": "NetworkTier"}
})
print(f"nameMapping: {nm.status_code} {nm.text[:80]}")
time.sleep(3)

# ── 2. Register SPD + DIR scheduled monitors ──────────────────────────────────
# Use /request endpoint (creates a background scheduled job).
# This is what the reference RHOAI 2.x demo uses and emits metrics to Prometheus.
# Plain numeric values — no TypedValue wrappers.
payload = {
    "modelId":             MODEL_NAME,
    "protectedAttribute":  "Region",
    "favorableOutcome":    2,       # Tier 2 (Premium) is favorable
    "outcomeName":         "NetworkTier",
    "privilegedAttribute": 0.0,     # Urban (Region_enc=0)
    "unprivilegedAttribute": 2.0,   # Rural (Region_enc=2)
    "batchSize":           5000
}

r = requests.post(f"{TRUSTYAI_INT}/metrics/group/fairness/spd/request",
                  headers=HEADERS, json=payload, timeout=30)
print(f"SPD: {r.status_code} {r.text[:150]}")

r2 = requests.post(f"{TRUSTYAI_INT}/metrics/group/fairness/dir/request",
                   headers=HEADERS, json=payload, timeout=30)
print(f"DIR: {r2.status_code} {r2.text[:150]}")

# ── 3. Compute baseline SPD & DIR manually + push to Prometheus ──────────────
print("\n── 3. Computing baseline SPD & DIR ──")
import numpy as np
_preds = clf.predict(X_test[FEATURES].values.astype(float))
_region = df.loc[X_test.index, "Region"]
_urban_mask = (_region == "Urban").values
_rural_mask  = (_region == "Rural").values
_p_urban = (_preds[_urban_mask] == 2).sum() / _urban_mask.sum()
_p_rural  = (_preds[_rural_mask]  == 2).sum() / _rural_mask.sum()
BASELINE_SPD = float(_p_urban - _p_rural)
BASELINE_DIR = float(_p_rural / _p_urban) if _p_urban > 0 else float("nan")

print(f"  Baseline SPD = {BASELINE_SPD:+.4f}  (fair if |SPD| < 0.10)")
print(f"  Baseline DIR = {BASELINE_DIR:.4f}   (fair if 0.80\u20131.20)")

# Push baseline to Prometheus via Pushgateway
PUSHGATEWAY_URL = f"http://pushgateway.{NAMESPACE}.svc.cluster.local:9091"
_payload = (
    f'trustyai_spd_live{{model="{MODEL_NAME}",step="baseline",protected="Region"}} {BASELINE_SPD}\n'
    f'trustyai_dir_live{{model="{MODEL_NAME}",step="baseline",protected="Region"}} {BASELINE_DIR}\n'
)
try:
    _r = requests.post(f"{PUSHGATEWAY_URL}/metrics/job/bias-monitor/model/{MODEL_NAME}",
        data=_payload, headers={"Content-Type": "text/plain"}, timeout=10)
    print(f"  \u2705 Baseline SPD/DIR pushed \u2192 Grafana shows {BASELINE_SPD:+.3f} (green baseline)")
except Exception as _e:
    print(f"  \u26a0\ufe0f  Push failed: {_e} (run Cell 8b first to deploy Pushgateway)")

> **📋 Demo Note — Cell 10 (Simulate Bias)** *(~30 seconds)*
>
> "Rural customers get \/month, urban get \/month — this pricing decision flows into the model as a proxy for Region. MonthlyCharges becomes a discriminatory signal even though Region isn't directly in the prediction."

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 10 — Simulate Bias (RHOAI 2.x)
# Purpose : Send a skewed batch via oc exec → agent → TrustyAI to push
#           SPD above the alert threshold, simulating model degradation.
#           Urban rows → high MonthlyCharges → model predicts Tier-2 more.
#           Rural rows  → low  MonthlyCharges → model predicts Tier-0/1.
# ══════════════════════════════════════════════════════════════════════════
import subprocess, requests, json, time, warnings
warnings.filterwarnings("ignore")

FEATURES = ["tenure","MonthlyCharges","TotalCharges","ContractType",
            "PaymentMethod","Region_enc","Income_enc"]

urban_rows = X_test[df.loc[X_test.index,"Region"]=="Urban"].head(100).copy()
rural_rows  = X_test[df.loc[X_test.index,"Region"]=="Rural"].head(100).copy()
urban_rows["MonthlyCharges"] = 115.0
rural_rows["MonthlyCharges"]  = 22.0
biased_batch = pd.concat([urban_rows, rural_rows])

# Get predictor pod for oc exec
_pred_pod = subprocess.run(
    ["oc","get","pod","-n",NAMESPACE,
     "-l",f"serving.kserve.io/inferenceservice={MODEL_NAME}",
     "-o","jsonpath={.items[0].metadata.name}"],
    capture_output=True, text=True).stdout.strip()
print(f"Sending biased inferences via: {_pred_pod}")

_sent = 0
for _, row in biased_batch.iterrows():
    _data = json.dumps({"inputs":[{"name":"float_input","shape":[1,7],"datatype":"FP32",
                                   "data":[float(v) for v in row[FEATURES].values]}]})
    _rc = subprocess.run(
        ["oc","exec","-n",NAMESPACE,_pred_pod,"-c","kserve-container","--",
         "curl","-s","--max-time","5","-X","POST",
         f"http://localhost:9081/v2/models/{MODEL_NAME}/infer",
         "-H","Content-Type: application/json","-d",_data,
         "-o","/dev/null","-w","%{http_code}"],
        capture_output=True, text=True)
    if _rc.stdout.strip() == "200":
        _sent += 1
    time.sleep(0.05)

print(f"Biased batch sent: {_sent}/200 inferences")
print("Watch Grafana / Observe > Metrics for SPD spike above 0.10")


> **📋 Demo Note — Cell 11 (Check SPD & DIR)** *(~20 seconds)*
>
> "TrustyAI gives us both the metric value and the direction of bias. SPD of +0.6 means Urban customers are 60% more likely to get Premium tier than Rural — that's discriminatory at scale."

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 11 — Check Current SPD & DIR Scores (RHOAI 2.x)
# Purpose : Query TrustyAI /q/metrics for both SPD and DIR.
#           Retrain triggers if EITHER metric is outside threshold.
# ══════════════════════════════════════════════════════════════════════════
import requests, subprocess, numpy as np

# Compute current SPD & DIR from model predictions (manual — accurate)
_preds = clf.predict(X_test[FEATURES].values.astype(float))
_region = df.loc[X_test.index, "Region"]
_urban_mask = (_region == "Urban").values
_rural_mask  = (_region == "Rural").values

# Use biased batch if available (from Cell 9), else use X_test predictions
try:
    _biased_preds  = clf.predict(biased_batch[FEATURES].values.astype(float))
    _biased_region = df.loc[biased_batch.index, "Region"]
    _u = (_biased_region == "Urban").values
    _r = (_biased_region == "Rural").values
    _p_u = (_biased_preds[_u] == 2).sum() / _u.sum() if _u.sum() > 0 else 0
    _p_r = (_biased_preds[_r] == 2).sum() / _r.sum() if _r.sum() > 0 else 0
    spd_val = float(_p_u - _p_r)
    dir_val = float(_p_r / _p_u) if _p_u > 0 else float("nan")
    print("Using biased_batch from Cell 9 for SPD/DIR computation")
except NameError:
    _p_u = (_preds[_urban_mask] == 2).sum() / _urban_mask.sum()
    _p_r = (_preds[_rural_mask]  == 2).sum() / _rural_mask.sum()
    spd_val = float(_p_u - _p_r)
    dir_val = float(_p_r / _p_u) if _p_u > 0 else float("nan")

print(f"{'═'*60}")
print(f"  Bias Metrics Summary (from model predictions)")
print(f"{'─'*60}")
print(f"  Baseline SPD : {BASELINE_SPD:+.4f}")
print(f"  Current SPD  : {spd_val:+.4f}  (threshold ±0.10)")
print(f"  Current DIR  : {dir_val:.4f}   (threshold 0.80–1.20)")
print(f"{'─'*60}")

spd_trigger = abs(spd_val) > 0.10
dir_trigger = not (0.8 <= dir_val <= 1.2) if not (dir_val != dir_val) else False
trigger     = spd_trigger or dir_trigger

print(f"  SPD trigger (|SPD| > 0.10) : {'YES ⚠️' if spd_trigger else 'NO ✅'}")
print(f"  DIR trigger (DIR < 0.80)   : {'YES ⚠️' if dir_trigger else 'NO ✅'}")
print(f"{'─'*60}")
print(f"  Retrain trigger (SPD OR DIR): {'YES — submitting retrain pipeline' if trigger else 'NO — within threshold'}")
print(f"{'═'*60}")

BIASED_SPD = spd_val
BIASED_DIR = dir_val

# Push biased values to Prometheus → Grafana spikes to red zone
PUSHGATEWAY_URL = f"http://pushgateway.{NAMESPACE}.svc.cluster.local:9091"
_payload = (
    f'trustyai_spd_live{{model="{MODEL_NAME}",step="biased",protected="Region"}} {spd_val}\n'
    f'trustyai_dir_live{{model="{MODEL_NAME}",step="biased",protected="Region"}} {dir_val}\n'
)
try:
    _r = requests.post(f"{PUSHGATEWAY_URL}/metrics/job/bias-monitor/model/{MODEL_NAME}",
        data=_payload, headers={"Content-Type": "text/plain"}, timeout=10)
    print(f"\n✅ Biased SPD/DIR pushed to Prometheus — Grafana now shows {spd_val:+.3f} (red zone)")
except Exception as _e:
    print(f"\n⚠️  Push failed: {_e}")


> **📋 Demo Note — Cell 11a (Retrain via KFP)**
>
> Use this cell if Data Science Pipelines is configured — demonstrates the full automated pipeline story. Skip to Cell 11b for a simpler local retrain.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 11a — Retrain Pipeline via KFP (requires Data Science Pipelines)
# ══════════════════════════════════════════════════════════════════════════
# RUN THIS if you want the full automated pipeline story with KFP.
# RUN Cell 11b instead if Data Science Pipelines is not yet configured.
#
# Triggers retrain if EITHER SPD > threshold OR DIR outside 0.80–1.20.
# ══════════════════════════════════════════════════════════════════════════
import subprocess, time, kfp
from kfp import dsl
from kfp.dsl import component

TOKEN = subprocess.check_output(["oc","whoami","-t"],
                                stderr=subprocess.DEVNULL).decode().strip()

# ── 1. Deploy DSPA (skip if already exists) ──────────────────────────────────
_dspa_exists = subprocess.run(
    ["oc","get","datasciencepipelinesapplication","dspa","-n",NAMESPACE],
    capture_output=True, text=True).returncode == 0

if not _dspa_exists:
    print("Deploying DataSciencePipelinesApplication…")
    _apply(f"""
apiVersion: datasciencepipelinesapplications.opendatahub.io/v1alpha1
kind: DataSciencePipelinesApplication
metadata:
  name: dspa
  namespace: {NAMESPACE}
spec:
  apiServer:
    deploy: true
  database:
    disableHealthCheck: false
    mariaDB:
      deploy: true
  objectStorage:
    externalStorage:
      bucket: {BUCKET_NAME}
      host: minio.{NAMESPACE}.svc.cluster.local:9000
      port: ""
      s3CredentialsSecret:
        accessKey: AWS_ACCESS_KEY_ID
        secretKey: AWS_SECRET_ACCESS_KEY
        secretName: aws-connection-minio-data-connection
      scheme: http
""")
else:
    print("DSPA already exists — skipping deployment")

# ── 2. Wait for KFP API server to be ready ───────────────────────────────────
print("\nWaiting for KFP API server (≤300s)…")
for _i in range(60):
    _phase = subprocess.run(
        ["oc","get","pod","-n",NAMESPACE,"-l","component=ds-pipeline-api-server",
         "-o","jsonpath={.items[0].status.phase}"],
        capture_output=True, text=True).stdout.strip()
    if _phase == "Running":
        print(f"  ✅ KFP API server Running"); break
    if _i % 4 == 0: print(f"  [{_i*5}s] phase={_phase!r}")
    time.sleep(5)
else:
    raise RuntimeError("KFP API server not Running after 300s")

time.sleep(10)
KFP_ENDPOINT = f"https://ds-pipeline-dspa-{NAMESPACE}.{CLUSTER_DOMAIN}"
print(f"KFP endpoint: {KFP_ENDPOINT}")

# ── 3. Define and submit KFP pipeline ────────────────────────────────────────
@component(base_image="python:3.11-slim",
           packages_to_install=["requests"])
def check_bias_threshold(trustyai_url: str, token: str,
                         spd_threshold: float,
                         dir_low: float, dir_high: float) -> bool:
    """Trigger retrain if SPD > threshold OR DIR outside [dir_low, dir_high]."""
    import requests
    headers = {"Authorization": f"Bearer {token}"}
    r = requests.get(f"{trustyai_url}/q/metrics", headers=headers, verify=False)
    spd_val, dir_val = None, None
    for line in r.text.split("\n"):
        if line.startswith("#"): continue
        if "trustyai_spd" in line:
            try: spd_val = float(line.strip().split(" ")[1])
            except: pass
        if "trustyai_dir" in line:
            try: dir_val = float(line.strip().split(" ")[1])
            except: pass
    spd_trigger = (spd_val is not None) and (abs(spd_val) > spd_threshold)
    dir_trigger = (dir_val is not None) and not (dir_low <= dir_val <= dir_high)
    print(f"SPD={spd_val} trigger={spd_trigger} | DIR={dir_val} trigger={dir_trigger}")
    return spd_trigger or dir_trigger

@component(base_image="python:3.11-slim",
           packages_to_install=["scikit-learn","pandas","numpy","skl2onnx","onnx"])
def retrain_model_fair(minio_endpoint: str, bucket: str, model_name: str) -> str:
    """Retrain on fairness-corrected data where Region does NOT affect Tier."""
    import numpy as np, pandas as pd
    from sklearn.neural_network import MLPClassifier
    from sklearn.model_selection import train_test_split
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType
    import onnx
    from onnx import helper, TensorProto

    np.random.seed(42); n = 7000
    df = pd.DataFrame({
        "tenure":         np.random.randint(1,72,n),
        "MonthlyCharges": np.random.uniform(20,120,n),
        "TotalCharges":   np.random.uniform(100,8000,n),
        "ContractType":   np.random.choice([0,1,2],n),
        "PaymentMethod":  np.random.choice([0,1,2,3],n),
        "Region":         np.random.choice(["Urban","Suburban","Rural"],n,p=[0.5,0.3,0.2]),
        "IncomeProxy":    np.random.choice(["High","Medium","Low"],n,p=[0.3,0.4,0.3])
    })
    def assign_tier_fair(row):
        score = row["MonthlyCharges"] / 120.0
        if row["IncomeProxy"] == "High":  score += 0.20
        elif row["IncomeProxy"] == "Low": score -= 0.10
        score += np.random.normal(0, 0.1)
        return 2 if score > 0.65 else (1 if score > 0.35 else 0)
    df["NetworkSliceTier"] = df.apply(assign_tier_fair, axis=1)
    df["Region_enc"] = df["Region"].map({"Urban":0,"Suburban":1,"Rural":2})
    df["Income_enc"] = df["IncomeProxy"].map({"High":0,"Medium":1,"Low":2})

    FEATURES = ["tenure","MonthlyCharges","TotalCharges","ContractType",
                "PaymentMethod","Region_enc","Income_enc"]
    X = df[FEATURES].astype(float); y = df["NetworkSliceTier"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    clf = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500, random_state=42)
    clf.fit(X_tr, y_tr)
    print(f"Fair model accuracy: {clf.score(X_te, y_te):.3f}")

    onnx_m = convert_sklearn(clf,
        initial_types=[("float_input", FloatTensorType([None,7]))], target_opset=12)
    new_nodes, zipmap_input = [], None
    for node in onnx_m.graph.node:
        if node.op_type == "ZipMap": zipmap_input = node.input[0]
        else: new_nodes.append(node)
    if zipmap_input:
        new_nodes.append(helper.make_node("ArgMax",
            inputs=[zipmap_input], outputs=["label_int64"], axis=1, keepdims=0))
        new_nodes.append(helper.make_node("Cast",
            inputs=["label_int64"], outputs=["label"], to=TensorProto.FLOAT))
        new_out = [helper.make_tensor_value_info("label", TensorProto.FLOAT, None)]
        g = helper.make_graph(new_nodes, onnx_m.graph.name,
                              list(onnx_m.graph.input), new_out,
                              list(onnx_m.graph.initializer))
        m2 = helper.make_model(g, opset_imports=onnx_m.opset_import)
        m2.ir_version = onnx_m.ir_version
        onnx.save(m2, "/tmp/retrained_fair_model.onnx")

    import boto3; from botocore.client import Config
    s3 = boto3.client("s3", endpoint_url=minio_endpoint,
                      aws_access_key_id="THEACCESSKEY",
                      aws_secret_access_key="THESECRETKEY",
                      config=Config(signature_version="s3v4"), verify=False)
    s3.upload_file("/tmp/retrained_fair_model.onnx", bucket,
                   f"{model_name}/1/model.onnx")
    print("✅ Fair model uploaded — ISVC will auto-reload")
    return "/tmp/retrained_fair_model.onnx"

@dsl.pipeline(name="bias-remediation-pipeline")
def bias_remediation_pipeline(
    trustyai_url: str = TRUSTYAI_INT,
    token: str = TOKEN,
    minio_endpoint: str = MINIO_ENDPOINT,
    bucket: str = BUCKET_NAME,
    model_name: str = MODEL_NAME,
    spd_threshold: float = 0.10,
    dir_low: float = 0.80,
    dir_high: float = 1.20
):
    check = check_bias_threshold(
        trustyai_url=trustyai_url, token=token,
        spd_threshold=spd_threshold, dir_low=dir_low, dir_high=dir_high)
    with dsl.If(check.output == True, name="bias-detected"):
        retrain_model_fair(
            minio_endpoint=minio_endpoint, bucket=bucket, model_name=model_name)

kfp.compiler.Compiler().compile(bias_remediation_pipeline, "/tmp/bias_pipeline.yaml")
client = kfp.Client(host=KFP_ENDPOINT, existing_token=TOKEN)
run = client.create_run_from_pipeline_package(
    "/tmp/bias_pipeline.yaml",
    arguments={"spd_threshold": 0.10, "dir_low": 0.80, "dir_high": 1.20},
    run_name="bias-remediation-run-01",
    experiment_name="telco-bias-monitoring")
print(f"✅ Pipeline submitted — Run ID: {run.run_id}")
print(f"   Track at: {KFP_ENDPOINT}/#/runs/details/{run.run_id}")


> **📋 Demo Note — Cell 11b (Retrain Locally)** *(~30 seconds)*
>
> "The fix is straightforward — retrain on fairness-corrected labels that don't encode geographic disadvantage. This runs directly in the notebook. In production you'd wire this to an automated pipeline."

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 11b — Retrain Fair Model Locally (no KFP required)
# Purpose : Retrains on fairness-corrected data directly in the notebook.
#           Use this when Data Science Pipelines is not yet configured.
# ══════════════════════════════════════════════════════════════════════════
import numpy as np, pandas as pd, boto3, warnings
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
from botocore.client import Config
import onnx
from onnx import helper, TensorProto
warnings.filterwarnings("ignore")

print("Step 1/4 — Generating fairness-corrected training data…")
np.random.seed(42); n = 7000
df_fair = pd.DataFrame({
    "tenure":         np.random.randint(1,72,n),
    "MonthlyCharges": np.random.uniform(20,120,n),
    "TotalCharges":   np.random.uniform(100,8000,n),
    "ContractType":   np.random.choice([0,1,2],n),
    "PaymentMethod":  np.random.choice([0,1,2,3],n),
    "Region":         np.random.choice(["Urban","Suburban","Rural"],n,p=[0.5,0.3,0.2]),
    "IncomeProxy":    np.random.choice(["High","Medium","Low"],n,p=[0.3,0.4,0.3])
})
def assign_tier_fair(row):
    # Region does NOT affect score — only charges + income matter
    score = row["MonthlyCharges"] / 120.0
    if row["IncomeProxy"] == "High":  score += 0.20
    elif row["IncomeProxy"] == "Low": score -= 0.10
    score += np.random.normal(0, 0.1)
    return 2 if score > 0.65 else (1 if score > 0.35 else 0)

df_fair["NetworkSliceTier"] = df_fair.apply(assign_tier_fair, axis=1)
df_fair["Region_enc"] = df_fair["Region"].map({"Urban":0,"Suburban":1,"Rural":2})
df_fair["Income_enc"] = df_fair["IncomeProxy"].map({"High":0,"Medium":1,"Low":2})
print(pd.crosstab(df_fair["Region"], df_fair["NetworkSliceTier"],
                  normalize="index").round(3))

print("\nStep 2/4 — Retraining MLP on fairness-corrected data…")
X_f = df_fair[FEATURES].astype(float)
y_f = df_fair["NetworkSliceTier"]
X_tr, X_te, y_tr, y_te = train_test_split(X_f, y_f, test_size=0.2, random_state=42)
clf_fair = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500, random_state=42)
clf_fair.fit(X_tr, y_tr)
print(f"  Fair model accuracy: {clf_fair.score(X_te, y_te):.3f}")

print("\nStep 3/4 — Exporting fair model to ONNX…")
onnx_m = convert_sklearn(clf_fair,
    initial_types=[("float_input", FloatTensorType([None,7]))], target_opset=12)
new_nodes, zipmap_input = [], None
for node in onnx_m.graph.node:
    if node.op_type == "ZipMap": zipmap_input = node.input[0]
    else: new_nodes.append(node)
if zipmap_input:
    new_nodes.append(helper.make_node("ArgMax",
        inputs=[zipmap_input], outputs=["label_int64"], axis=1, keepdims=0))
    new_nodes.append(helper.make_node("Cast",
        inputs=["label_int64"], outputs=["label"], to=TensorProto.FLOAT))
    new_out = [helper.make_tensor_value_info("label", TensorProto.FLOAT, None)]
    g = helper.make_graph(new_nodes, onnx_m.graph.name,
                          list(onnx_m.graph.input), new_out,
                          list(onnx_m.graph.initializer))
    m2 = helper.make_model(g, opset_imports=onnx_m.opset_import)
    m2.ir_version = onnx_m.ir_version
    onnx.save(m2, "/tmp/retrained_fair_model.onnx")
print("  Saved to /tmp/retrained_fair_model.onnx")

print("\nStep 4/4 — Uploading fair model to MinIO…")
s3 = boto3.client("s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version="s3v4"), verify=False)
s3.upload_file("/tmp/retrained_fair_model.onnx", BUCKET_NAME,
               f"{MODEL_NAME}/1/model.onnx")
print(f"  ✅ Uploaded → s3://{BUCKET_NAME}/{MODEL_NAME}/1/model.onnx")
print("\n✅ Fair model deployed — ISVC will auto-reload → run Cell 12")


> **📋 Demo Note — Cell 12 — CLOSE STRONG** *(~30 seconds)*
>
> "After retraining, SPD drops from +0.6 back to ~+0.02 — well within the acceptable range. Grafana shows the green zone restored. TrustyAI detected the bias, quantified it, and confirmed the fix — all automated, all auditable, all in OpenShift AI."
>
> **Closing line:** *"TrustyAI gives telecom operators continuous, automated fairness guardrails on every model in production — not just a one-time check at launch."*

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 12 — Verify SPD & DIR Reduced After Retraining (RHOAI 2.x)
# Purpose : Compute both metrics with the retrained fair model and confirm
#           both have returned within their thresholds.
#           Also sends fresh inferences so TrustyAI /q/metrics reflects
#           the fair model.
# ══════════════════════════════════════════════════════════════════════════
import numpy as np, requests, subprocess, time, warnings
warnings.filterwarnings("ignore")

TOKEN   = subprocess.check_output(["oc","whoami","-t"], stderr=subprocess.DEVNULL).decode().strip()
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

# ── Local SPD & DIR check ─────────────────────────────────────────────────────
try:
    clf_fair
    print("Using clf_fair from Cell 11b")
    def compute_fair_preds(X_sample):
        return clf_fair.predict(X_sample[FEATURES].values.astype(float))
except NameError:
    import onnxruntime as rt
    fair_sess = rt.InferenceSession("/tmp/retrained_fair_model.onnx")
    print("Using ONNX model from /tmp/retrained_fair_model.onnx")
    def compute_fair_preds(X_sample):
        inputs = X_sample[FEATURES].values.astype(np.float32)
        return fair_sess.run(None, {"float_input": inputs})[0].flatten()

preds_fair = compute_fair_preds(X_test)
region     = df.loc[X_test.index, "Region"]
urban_mask = (region == "Urban").values
rural_mask = (region == "Rural").values

fav_urban = (preds_fair[urban_mask] == 2).sum()
fav_rural  = (preds_fair[rural_mask]  == 2).sum()
p_urban   = fav_urban / urban_mask.sum()
p_rural    = fav_rural  / rural_mask.sum()
fair_spd  = float(p_urban - p_rural)
fair_dir  = float(p_rural / p_urban) if p_urban > 0 else float("nan")

fair_result = dict(spd=fair_spd, dir=fair_dir,
                   p_urban=float(p_urban), p_rural=float(p_rural),
                   n_urban=int(urban_mask.sum()), n_rural=int(rural_mask.sum()),
                   fav_urban=int(fav_urban), fav_rural=int(fav_rural))

# Print bias report inline (no external function needed)
print(f"{'='*60}")
print(f"  Post-Retrain Bias Analysis (Fair Model)")
print(f"  Protected Attribute : Region (Urban vs Rural)")
print(f"  Favorable Outcome   : Tier 2 (Premium Network Slice)")
print(f"{'-'*60}")
print(f"  Urban  : {fair_result['p_urban']:.4f}  ({fair_result['fav_urban']}/{fair_result['n_urban']} Premium)")
print(f"  Rural  : {fair_result['p_rural']:.4f}  ({fair_result['fav_rural']}/{fair_result['n_rural']} Premium)")
print(f"  SPD    : {fair_result['spd']:+.4f}  (target |SPD| < 0.10)")
print(f"  DIR    : {fair_result['dir']:.4f}  (target 0.80–1.20)")
print(f"{'='*60}")

# ── Full journey summary ──────────────────────────────────────────────────────
print(f"\n{'═'*60}")
print(f"  Bias Metrics Journey")
print(f"{'─'*60}")
print(f"  {'Metric':<8}  {'Baseline':>10}  {'After Bias':>10}  {'After Retrain':>14}  Status")
print(f"{'─'*60}")

spd_ok = abs(fair_spd) <= 0.10
dir_ok = 0.8 <= fair_dir <= 1.2
print(f"  {'SPD':<8}  {BASELINE_SPD:>+10.4f}  {BIASED_SPD:>+10.4f}  {fair_spd:>+14.4f}  {'✅' if spd_ok else '⚠️'}")
print(f"  {'DIR':<8}  {BASELINE_DIR:>10.4f}  {BIASED_DIR:>10.4f}  {fair_dir:>14.4f}  {'✅' if dir_ok else '⚠️'}")
print(f"{'─'*60}")

both_ok   = spd_ok and dir_ok
either_ok = spd_ok or dir_ok
if both_ok:
    print(f"  ✅ Both SPD and DIR within threshold — fairness fully restored")
elif either_ok:
    print(f"  ⚠️  Partial improvement — one metric still outside threshold")
    print(f"      Consider additional retraining iterations")
else:
    print(f"  ⚠️  Neither metric improved — check retrain pipeline")
print(f"{'═'*60}")

# ── Send fresh inferences + check TrustyAI /q/metrics ────────────────────────
print("\nSending 100 fresh inferences so TrustyAI reflects the fair model…")
_sent = 0
for _, _row in X_test.head(100).iterrows():
    _payload = {"inputs":[{"name":"float_input","shape":[1,7],"datatype":"FP32",
                            "data":[float(v) for v in _row[FEATURES].values]}]}
    try:
        _r = requests.post(INFER_URL, headers=HEADERS, json=_payload,
                           timeout=10, verify=False)
        if _r.status_code == 200: _sent += 1
    except Exception: pass
    time.sleep(0.05)
print(f"  {_sent}/100 sent")

time.sleep(15)
r = requests.get(f"{TRUSTYAI_INT}/q/metrics", headers=HEADERS, timeout=10)
print("\nTrustyAI /q/metrics after retrain:")
for line in r.text.split("\n"):
    if ("trustyai_spd" in line or "trustyai_dir" in line) and not line.startswith("#"):
        print(line)

# Push fair values to Prometheus → Grafana drops back to green
PUSHGATEWAY_URL = f"http://pushgateway.{NAMESPACE}.svc.cluster.local:9091"
_payload = (
    f'trustyai_spd_live{{model="{MODEL_NAME}",step="fair",protected="Region"}} {fair_spd}\n'
    f'trustyai_dir_live{{model="{MODEL_NAME}",step="fair",protected="Region"}} {fair_dir}\n'
)
try:
    _r = requests.post(f"{PUSHGATEWAY_URL}/metrics/job/bias-monitor/model/{MODEL_NAME}",
        data=_payload, headers={"Content-Type": "text/plain"}, timeout=10)
    print(f"✅ Fair SPD/DIR pushed to Prometheus — Grafana drops to {fair_spd:+.3f} (green zone)")
except Exception as _e:
    print(f"⚠️  Push failed: {_e}")
